# Evaluación del agente PISA

Este notebook pregunta al agente los casos de `reference_test.py` y evalúa cada respuesta de tres formas:

* **Herramientas** (automática): si llamó a las herramientas imprescindibles del caso.
* **Estilo** (automática): si la respuesta contiene tecnicismos que el System Prompt prohíbe, como nombres de variables (`recursos_estudio`), códigos de perfil o país (`PUB_URB_MED`, `AUS`), nombres de herramientas o el peso `SENWT`. Nombrar la regresión, el XGBoost o SHAP está permitido.
* **Fidelidad, calidad y completitud** (juez): un LLM de otra familia que el agente (`qwen/qwen3.8-27b`) compara la respuesta con las cifras y conclusiones de referencia (`judge_prompt.md`).

**Cómo leer los resultados.**

* `Fidelidad OK = FALLO`: la respuesta inventa o atribuye mal una cifra, o contradice una conclusión esperada.
* `Respaldo = True`: respondió, al menos en parte, `gpt-oss-20b` porque el modelo principal no estaba disponible. Ese modelo lee peor las tablas, así que un fallo con respaldo puede no repetirse con el principal.
* `ERROR`: falló la llamada (cuota de Groq, límite de pasos del agente...), no la respuesta. Las medias de calidad y completitud no cuentan esos casos.

**LangSmith.** Si `LANGSMITH_TRACING=true` en el `.env`, cada caso deja una traza «eval · <caso>» y el juez otra «juez · <caso>», con las etiquetas `evaluacion` y el identificador de la ejecución. Ahí se ve qué herramientas llamó el agente, con qué argumentos y qué le devolvieron: sirve para entender por qué falla un caso. Las puntuaciones solo están en este notebook y en los CSV de `resultados/`.

## 0. Configuración

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

# El agente importa config, herramientas... desde agente_pisa/
DIR_AGENTE = Path.cwd().parent if Path.cwd().name == "evaluador" else Path.cwd()
sys.path.insert(0, str(DIR_AGENTE))

from agente import construir_agente
from evaluador import (CASOS_TEST_PISA, MODELO_JUEZ, cargar_resultados, comparar_ejecuciones, ejecutar_bateria_evaluacion,
                       guardar_resultados, resumen_evaluacion)
from evaluador.graficos_evaluacion import grafico_comparacion, grafico_criterios, grafico_puntuaciones

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

## 1. Casos que se van a evaluar

Por defecto, todos. Para repetir solo algunos (por ejemplo, los que fallaron), filtra `CASOS` por `id` o por `categoria`. `ETIQUETA` describe la ejecución (qué cambiaste antes de lanzarla) y sirve para compararla después con otras.

In [ ]:
CASOS = CASOS_TEST_PISA
# CASOS = [caso for caso in CASOS_TEST_PISA if caso["categoria"] == "Robustez"]
# CASOS = [caso for caso in CASOS_TEST_PISA if caso["id"] in {"efecto_techo_recursos", "xgboost_vs_regresion"}]
ETIQUETA = ""          # por ejemplo: "tras corregir modelos.md"
PAUSA_SEG = 0          # segundos entre casos, si Groq da errores de cuota

display(pd.DataFrame([{"ID": caso["id"], "Categoría": caso["categoria"], "Pregunta": caso["pregunta"],
                       "Herramientas imprescindibles": ", ".join(caso["herramientas_imprescindibles"]) or "Ninguna"}
                      for caso in CASOS]))
print(f"{len(CASOS)} casos · juez: {MODELO_JUEZ}")

## 2. Ejecutar la evaluación

Cada caso se pregunta en un hilo nuevo, sin memoria de los anteriores. Los resultados se guardan en `evaluador/resultados/evaluacion_<fecha>_<hora>.csv`.

In [ ]:
agente = construir_agente()
resultados = ejecutar_bateria_evaluacion(agente, casos=CASOS, etiqueta=ETIQUETA, pausa_seg=PAUSA_SEG)
ruta = guardar_resultados(resultados)
print(f"Guardado en {ruta.relative_to(DIR_AGENTE)}")

## 3. Resumen

Porcentaje de casos con `OK` en cada comprobación y puntuaciones medias del juez, por categoría y en total.

In [ ]:
display(resumen_evaluacion(resultados))

In [ ]:
grafico_criterios(resultados)
plt.show()

In [ ]:
grafico_puntuaciones(resultados)
plt.show()

## 4. Casos con algún fallo

Para cada caso con `FALLO`, `AVISO` o `ERROR`: qué herramientas esperaba y usó, el modelo que respondió, el dictamen del juez y la respuesta completa del agente. La traza de LangSmith del mismo caso muestra lo que devolvieron las herramientas.

In [ ]:
CRITERIOS = ["Tools OK", "Fidelidad OK", "Estilo Limpio"]
con_fallo = resultados[resultados[CRITERIOS].ne("OK").any(axis=1)]
print(f"{len(con_fallo)} de {len(resultados)} casos con algún fallo")
for _, caso in con_fallo.iterrows():
    estado = " · ".join(f"{criterio}: {caso[criterio]}" for criterio in CRITERIOS)
    modelo = caso["Modelo"] + (" (respaldo)" if caso["Respaldo"] else "")
    tecnicismos = f" · **Tecnicismos:** {caso['Tecnicismos']}" if caso["Tecnicismos"] else ""
    respuesta = "> " + str(caso["Respuesta Agente"]).replace("\n", "\n> ")
    display(Markdown("\n\n".join([
        f"### {caso['ID']}",
        f"**Pregunta:** {caso['Pregunta']}",
        f"**{estado}** · calidad {caso['Calidad (1-5)']} · completitud {caso['Completitud (1-5)']}",
        f"**Herramientas:** esperadas {caso['Tools Esperadas']}; usadas {caso['Tools Usadas']} · **Modelo:** {modelo}{tecnicismos}",
        f"**Juez:** {caso['Dictamen Juez']}",
        "**Respuesta del agente:**",
        respuesta,
    ])))

## 5. Comparar con otra ejecución

Compara caso a caso dos ejecuciones guardadas: por defecto, la anterior y la última. Sirve para ver si un cambio en el System Prompt, las notas de metodología o las herramientas arregla unos casos sin estropear otros. Solo se comparan los casos que están en las dos.

In [ ]:
historico = cargar_resultados()
ejecuciones = (historico.groupby("Ejecución", sort=True)
               .agg(Etiqueta=("Etiqueta", "first"), Casos=("ID", "size"),
                    Fidelidad_OK=("Fidelidad OK", lambda x: f"{(x == 'OK').mean():.0%}"),
                    Calidad_media=("Calidad (1-5)", "mean")))
display(ejecuciones)

In [ ]:
if len(ejecuciones) < 2:
    print("Hace falta al menos otra ejecución guardada para comparar.")
else:
    ANTERIOR, ACTUAL = ejecuciones.index[-2], ejecuciones.index[-1]   # o escribe los identificadores
    comparacion = comparar_ejecuciones(historico, ANTERIOR, ACTUAL)
    print(f"{ANTERIOR} -> {ACTUAL}: {int(comparacion[('Cambia', '')].sum())} de {len(comparacion)} casos cambian")
    display(comparacion)
    grafico_comparacion(comparacion, "Calidad (1-5)")
    plt.show()